In [1]:
# install bio package quietly
!pip install biopython --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 24.9 MB/s eta 0:00:00


# Imports

In [2]:
# Linear algebra
import numpy as np

# Dataframes
import pandas as pd

# Path
import os

# Translate codons into amino acids
from Bio.Seq import translate

# save json file
import json

# Tokenizer:
  # Pre-Tokenizer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.models import WordLevel
  # Tokenizer
from tokenizers import Tokenizer
  # Post-Tokenizer
from tokenizers.processors import TemplateProcessing
  # Wrapper
from transformers import PreTrainedTokenizerFast
  # Load tokenizer
from transformers import AutoTokenizer

# Generate tokens

## Utils

**Function that returns all possible codons:**

In [30]:
def get_dna_codons():
  nt_bases = ['A', 'C', 'G', 'T']
  codons = [c1 + c2 + c3 for c1 in nt_bases for c2 in nt_bases for c3 in nt_bases]
  return codons

**Function that returns all tokens used by the Tokenizer:**

In [31]:
def get_tokenset():
  """Returns all tokens used by the tokenizer."""

  # get all possible codons
  codons = get_dna_codons()

  # count all unique amino acids (+ stop codon) and generate codon masks
  _unique_aa = set([translate(c) for c in codons])
  mask_tokens = ['<mask_'+aa+'>' for aa in _unique_aa]

  # format tokens
  _format_tokens = ["<pad>", "<s>", "</s>", "<unk>", "<cls>", "<sep>"]

  # tokens for the expression vectors
  _expr_tokens = ["<expr_top10>", "<expr_pre75_90>", "<expr_pre50_75>", "<expr_pre25_50>", "<expr_low25>", "<expr_unk>"]

  # unify all special tokens
  special_tokens = [*_format_tokens, *_expr_tokens]

  return codons, mask_tokens, special_tokens

## Generate the tokens

In [32]:
codons, mask_tokens, special_tokens = get_tokenset()

# Generate Tokenizer

## Initialize Tokenizer and Pre-Processing

**Instantiate WordLevel Tokenizer Object**

In [33]:
tokenizer = Tokenizer(WordLevel(unk_token="<unk>"))

**Pre-Tokenizer**

In [34]:
# Separate codons by splitting on whitespace
tokenizer.pre_tokenizer = Whitespace()

**Add tokens to vocabulary**

In [35]:
# Add codon tokens
tokenizer.add_tokens(codons)

64

**Add special tokens to vocabulary**

In [36]:
# add mask tokens
tokenizer.add_special_tokens(mask_tokens)

21

In [37]:
# add other special tokens
tokenizer.add_special_tokens(special_tokens)

12

## Post-Processing

**Post processing**

In [38]:
# Add special tokens used in post processing
cls_token_id = tokenizer.token_to_id("<cls>")
sep_token_id = tokenizer.token_to_id("<sep>")

# Post processing sentece template
tokenizer.post_processor = TemplateProcessing(
        single=f"<cls> $A <sep>",
        pair=f"<cls> $A <sep> $B:1 <sep>:1",
        special_tokens=[("<cls>", cls_token_id), ("<sep>", sep_token_id)],
        )

## Wrap Tokenizer

**Initialize Wrapper**

In [39]:
wrapped_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object = tokenizer,

    bos_token = "<s>",
    eos_token = "</s>",
    unk_token = "<unk>",
    pad_token = "<pad>",
    cls_token = "<cls>",
    sep_token = "<sep>",
    model_max_length = 512,
    padding_side = "right",
    clean_up_tokenization_spaces = True
    )

# Save Tokenizer Files

## Tokenizer Directory

In [40]:
tokenizer_path = "./tokenizer/"

## Save Wrapped Tokenizer

In [41]:
wrapped_tokenizer.save_pretrained(tokenizer_path)

('./tokenizer/tokenizer_config.json',
 './tokenizer/special_tokens_map.json',
 './tokenizer/tokenizer.json')

## Save Masking Dictionary

In [42]:
# translates codons to aa mask
mask_translator_dict = {wrapped_tokenizer.vocab[c] : wrapped_tokenizer.vocab['<mask_' + translate(c) + '>'] for c in codons}
# special tokens translated to (remain) themselves
mask_translator_dict = {**mask_translator_dict,
                        **{wrapped_tokenizer.vocab[m] : wrapped_tokenizer.vocab[m] for m in mask_tokens}}
mask_translator_dict = {**mask_translator_dict,
                        **{wrapped_tokenizer.vocab[s] : wrapped_tokenizer.vocab[s] for s in special_tokens}}

In [43]:
with open(os.path.join(tokenizer_path, "masking_dict.json"), "w") as f:
    json.dump(mask_translator_dict, f)

## Restrict prediction for each token

In [44]:
tokenizer = wrapped_tokenizer

# initialize an empty list for each mask token
mask_restriction_dict = {}
for mask in mask_tokens:
    mask_restriction_dict[wrapped_tokenizer.vocab[mask]] = []

# add all corresponding codons to each mask list
for c in codons:
    # restrict masked aa to its codons
    maskid = wrapped_tokenizer.vocab["<mask_" + translate(c) + ">"]
    mask_restriction_dict[maskid] += [wrapped_tokenizer.vocab[c]]

    # restrict real codon to itself
    mask_restriction_dict[wrapped_tokenizer.vocab[c]] = [wrapped_tokenizer.vocab[c]]

# restrict other special tokens to themselves
for s in special_tokens:
    mask_restriction_dict[wrapped_tokenizer.vocab[s]] = [wrapped_tokenizer.vocab[s]]

# remove list duplicates
for k in mask_restriction_dict.keys():
    mask_restriction_dict[k] = list(set(mask_restriction_dict[k]))

### Save Restriction Dictionary

In [45]:
# Save tokenizer
with open(os.path.join(tokenizer_path,'mask_restrict_dict.json'), 'w') as f:
    json.dump(mask_restriction_dict, f)

# Load and Try Tokenizer

**Load tokenizer**

In [48]:
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
print(tokenizer)
print(mask_translator_dict)

PreTrainedTokenizerFast(name_or_path='./tokenizer/', vocab_size=0, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '<sep>', 'pad_token': '<pad>', 'cls_token': '<cls>'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("AAA", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	1: AddedToken("AAC", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	2: AddedToken("AAG", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	3: AddedToken("AAT", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	4: AddedToken("ACA", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	5: AddedToken("ACC", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	6: AddedToken("ACG", rstrip=Fal

In [47]:
tokenizer('</s>')

{'input_ids': [89, 87, 90], 'token_type_ids': [0, 0, 0], 'attention_mask': [1, 1, 1]}

In [54]:
len(tokenizer)

97

**Try tokenizer on a list of sequences**

In [49]:
seq = ["AAA AAG ACG CGC GGG AAA AAA AAA", "AGG ACG ATT"]
other_seq = ["AAA AAG ACG CGC GGG AAA AAA AAA", "AGG ACG ATT"]

# print(tokenizer.vocab)
print(tokenizer(seq, other_seq, return_tensors = 'pt', padding = True)['input_ids'])
print(tokenizer(seq, other_seq, return_tensors = 'pt', padding = True)['attention_mask'])
# ids = tokenizer(seq, return_tensors = 'pt', padding = True).input_ids
# print(tokenizer.decode(ids[1]))

tensor([[89,  0,  2,  6, 25, 42,  0,  0,  0, 90,  0,  2,  6, 25, 42,  0,  0,  0,
         90],
        [89, 10,  6, 15, 90, 10,  6, 15, 90, 85, 85, 85, 85, 85, 85, 85, 85, 85,
         85]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])


In [50]:
bool_arr = [True, False, True, False]
int_arr = [0, 1]

np.where(bool_arr[:2], int_arr, -100)

array([   0, -100])

In [51]:
tokenizer.sep_token_id

90

In [52]:
seq = ['ATG ACG AAG TAA', "ATG TGA"]
batch = tokenizer(seq, return_tensors = 'pt', padding = True)
ids = batch['input_ids']
print(ids)
print(tokenizer.decode(ids[0]))
print(batch['attention_mask'])

tensor([[89, 14,  6,  2, 48, 90],
        [89, 14, 56, 90, 85, 85]])
<cls> ATG ACG AAG TAA <sep>
tensor([[1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 0, 0]])


In [53]:
query_dna_sequences = ["ATG AGG AGT AAA", "ATG TGA AGT ATG"]
subject_dna_sequences = ["ATG AGG AGT AAA", "ATG TGA"]
batch = tokenizer(query_dna_sequences, subject_dna_sequences, return_tensors = 'pt', padding = "longest")